# Flash Attention: From Theory to Practice

This notebook explores **Flash Attention** (Dao et al., 2022), an IO-aware exact attention
algorithm that reduces memory usage from $O(N^2)$ to $O(N)$ and achieves 2-4x wall-clock speedup
over standard attention by exploiting the GPU memory hierarchy.

We will:
1. Review why standard attention is slow (it's memory-bound, not compute-bound)
2. Implement naive scaled dot-product attention from scratch
3. Implement the Flash Attention forward pass in pure PyTorch (tiled, with online softmax)
4. Use PyTorch's built-in `torch.nn.functional.scaled_dot_product_attention`
5. Benchmark all three approaches for runtime and memory
6. Visualize the results

**Reference:** Dao, T., Fu, D.Y., Ermon, S., Rudra, A., & Re, C. (2022).
*FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness.* NeurIPS 2022.
https://arxiv.org/abs/2205.14135

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn.functional as F
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.device import set_device, set_seed

device = set_device()
set_seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. The Memory Hierarchy Problem

GPUs have two main memory tiers:

```
                      GPU
    ┌─────────────────────────────────────┐
    │                                     │
    │   ┌─────────────────────────────┐   │
    │   │  SRAM (On-Chip)             │   │
    │   │  192 KB per SM              │   │
    │   │  ~19 TB/s bandwidth         │   │
    │   │  (very fast, very small)    │   │
    │   └─────────────────────────────┘   │
    │               ↕ data movement       │
    │   ┌─────────────────────────────┐   │
    │   │  HBM (High Bandwidth Mem)   │   │
    │   │  40-80 GB                   │   │
    │   │  1.5-2.0 TB/s bandwidth    │   │
    │   │  (slower, much larger)      │   │
    │   └─────────────────────────────┘   │
    │                                     │
    └─────────────────────────────────────┘
                (NVIDIA A100)
```

**Key insight:** SRAM is ~10x faster than HBM, but ~200,000x smaller.

Standard attention is **memory-bound**, not compute-bound. The bottleneck isn't the
matrix multiplications — it's reading and writing the $N \times N$ attention matrix
to and from HBM. Every time we create that matrix, we pay a round-trip to slow memory.

## 2. Standard Attention

The standard scaled dot-product attention:

$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}}\right) V$

where $Q, K, V \in \mathbb{R}^{N \times d}$.

**Memory cost:** The score matrix $S = QK^T / \sqrt{d}$ has shape $N \times N$. For $N = 4096$:

$4096 \times 4096 \times 4 \text{ bytes (float32)} = 64 \text{ MB per head per batch element}$

Standard attention requires **4 HBM round trips** for one attention operation:
1. Write $S = QK^T$ to HBM ($N^2$)
2. Read $S$ for softmax, write $P = \text{softmax}(S)$ to HBM ($N^2$)
3. Read $P$ for the final matmul with $V$ ($N^2$)

**Total HBM accesses:** $O(N^2 d + N^2) = O(N^2 d)$

In [ ]:
def naive_attention(Q, K, V):
    """Standard scaled dot-product attention. O(N^2) memory."""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # [B, N, N] <- the bottleneck
    attn_weights = torch.softmax(scores, dim=-1)         # [B, N, N]
    output = attn_weights @ V                             # [B, N, d]
    return output


# Demo on a small example
B, N, d = 1, 4, 2
Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)

scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
print(f"Q shape:      {Q.shape}")
print(f"K shape:      {K.shape}")
print(f"Scores shape: {scores.shape}  <- this is the N x N matrix")
print(f"\nFor N=4096, d=64, this matrix would be:")
print(f"  {4096*4096*4 / 1024**2:.0f} MB per head per batch element")

## 3. The Flash Attention Insight

Flash Attention avoids materializing the $N \times N$ matrix using two key ideas:

### Idea 1: Tiling

Instead of computing the full $S = QK^T$, we split $Q$, $K$, $V$ into blocks and process
one block pair at a time. Each block's score matrix is only $B_r \times B_c$ — small enough
to fit in SRAM.

### Idea 2: Online Softmax

Softmax normally needs the full row of scores to compute the denominator. Flash Attention
uses an **online** (incremental) softmax that maintains running statistics across blocks:

- **Running max** $m_i$: tracks the maximum score seen so far for row $i$
- **Running sum** $\ell_i$: tracks the softmax denominator for row $i$

When processing a new block $j$, we update:

$m_i^{\text{new}} = \max(m_i, \tilde{m}_{ij})$

$\ell_i^{\text{new}} = e^{m_i - m_i^{\text{new}}} \cdot \ell_i + e^{\tilde{m}_{ij} - m_i^{\text{new}}} \cdot \tilde{\ell}_{ij}$

$O_i \leftarrow \frac{e^{m_i - m_i^{\text{new}}} \cdot \ell_i \cdot O_i + e^{\tilde{m}_{ij} - m_i^{\text{new}}} \cdot \tilde{P}_{ij} V_j}{\ell_i^{\text{new}}}$

where $\tilde{m}_{ij} = \text{rowmax}(S_{ij})$, $\tilde{P}_{ij} = \exp(S_{ij} - \tilde{m}_{ij})$, and $\tilde{\ell}_{ij} = \text{rowsum}(\tilde{P}_{ij})$.

**Why this works:** The softmax denominator is a sum of exponentials. By tracking the running
max and rescaling with $e^{m_{\text{old}} - m_{\text{new}}}$, we correctly adjust previously
accumulated values when the max changes.

## 4. Algorithm 1: Flash Attention Forward Pass

**Input:** $Q, K, V \in \mathbb{R}^{N \times d}$, block sizes $B_r, B_c$

1. Initialize $O = 0 \in \mathbb{R}^{N \times d}$, $\ell = 0 \in \mathbb{R}^{N}$, $m = -\infty \in \mathbb{R}^{N}$
2. Split $Q$ into row blocks of size $B_r$, and $K$, $V$ into column blocks of size $B_c$
3. **For each** column block $j$ of $K, V$:
4. &emsp; **For each** row block $i$ of $Q$:
5. &emsp;&emsp; Compute $S_{ij} = Q_i K_j^T / \sqrt{d}$ &emsp; *(only $B_r \times B_c$, fits in SRAM)*
6. &emsp;&emsp; Compute $\tilde{m}_{ij} = \text{rowmax}(S_{ij})$
7. &emsp;&emsp; Compute $\tilde{P}_{ij} = \exp(S_{ij} - \tilde{m}_{ij})$
8. &emsp;&emsp; Compute $\tilde{\ell}_{ij} = \text{rowsum}(\tilde{P}_{ij})$
9. &emsp;&emsp; Update $m_i, \ell_i, O_i$ using the online softmax rules above
10. **Return** $O$

**Block sizes** (from SRAM size $M$ and head dimension $d$):

$B_c = \left\lceil \frac{M}{4d} \right\rceil, \quad B_r = \min\!\left(\left\lceil \frac{M}{4d} \right\rceil, d\right)$

The factor of 4 accounts for $Q_i$, $K_j$, $V_j$, and $O_i$ all needing to fit in SRAM simultaneously.

**IO Complexity:**
- Standard attention: $O(N^2 d)$ HBM accesses
- Flash Attention: $O(N^2 d^2 / M)$ HBM accesses
- Since $d \ll M$ typically, this is a significant reduction

In [ ]:
def flash_attention(Q, K, V, block_size=32):
    """
    Flash Attention forward pass with tiled online softmax.
    
    Implements Algorithm 1 from Dao et al. (2022).
    Processes Q, K, V in blocks to avoid materializing the full N x N matrix.
    
    NOTE: This pure-PyTorch implementation demonstrates the algorithm but
    won't match CUDA kernel speed. See Cell 10 for why.
    """
    B, N, d = Q.shape
    B_r = block_size
    B_c = block_size
    scale = 1.0 / math.sqrt(d)

    # Step 1: Initialize output O, running sum l, running max m
    O = torch.zeros_like(Q)                                    # [B, N, d]
    l = torch.zeros(B, N, 1, device=Q.device, dtype=Q.dtype)  # [B, N, 1]
    m = torch.full((B, N, 1), float("-inf"), device=Q.device, dtype=Q.dtype)  # [B, N, 1]

    # Step 2: Determine number of blocks
    num_col_blocks = math.ceil(N / B_c)
    num_row_blocks = math.ceil(N / B_r)

    # Step 3: Outer loop over K, V column blocks
    for j in range(num_col_blocks):
        j_start = j * B_c
        j_end = min(j_start + B_c, N)
        K_j = K[:, j_start:j_end, :]  # [B, B_c, d]
        V_j = V[:, j_start:j_end, :]  # [B, B_c, d]

        # Step 4: Inner loop over Q row blocks
        for i in range(num_row_blocks):
            i_start = i * B_r
            i_end = min(i_start + B_r, N)
            Q_i = Q[:, i_start:i_end, :]

            # Load current running statistics
            m_i = m[:, i_start:i_end, :]
            l_i = l[:, i_start:i_end, :]
            O_i = O[:, i_start:i_end, :]

            # Step 5: Compute block scores (only B_r x B_c, NOT N x N)
            S_ij = (Q_i @ K_j.transpose(-2, -1)) * scale  # [B, B_r, B_c]

            # Step 6: Block row max
            m_ij = S_ij.max(dim=-1, keepdim=True).values  # [B, B_r, 1]

            # Step 7: Block softmax numerator
            P_ij = torch.exp(S_ij - m_ij)  # [B, B_r, B_c]

            # Step 8: Block row sum
            l_ij = P_ij.sum(dim=-1, keepdim=True)  # [B, B_r, 1]

            # Step 9: Online softmax update
            m_new = torch.maximum(m_i, m_ij)
            l_new = torch.exp(m_i - m_new) * l_i + torch.exp(m_ij - m_new) * l_ij

            # Rescale accumulated output and add new contribution
            O_new = (
                torch.exp(m_i - m_new) * l_i * O_i
                + torch.exp(m_ij - m_new) * (P_ij @ V_j)
            ) / l_new

            # Write back
            m[:, i_start:i_end, :] = m_new
            l[:, i_start:i_end, :] = l_new
            O[:, i_start:i_end, :] = O_new

    return O


print("Flash attention implementation ready.")

In [ ]:
# Correctness verification: flash attention must match naive attention exactly
B, N, d = 2, 64, 32
Q = torch.randn(B, N, d, device=device)
K = torch.randn(B, N, d, device=device)
V = torch.randn(B, N, d, device=device)

out_naive = naive_attention(Q, K, V)
out_flash = flash_attention(Q, K, V, block_size=16)

max_diff = (out_naive - out_flash).abs().max().item()
matches = torch.allclose(out_naive, out_flash, atol=1e-5)

print(f"Max absolute difference: {max_diff:.2e}")
print(f"Outputs match (atol=1e-5): {matches}")
assert matches, "Flash attention output does not match naive attention!"

## 5. Why Won't Our Pure-Python Flash Attention Be Faster?

The speedup in the original Flash Attention comes from writing a **custom CUDA kernel** that
keeps tile computations entirely in SRAM. Our Python implementation still dispatches
individual PyTorch operations to HBM — each `Q_i @ K_j.T` call goes through the standard
PyTorch dispatch, reading from and writing to HBM.

The *algorithmic* benefits are real:
- **O(N) memory** (no N x N materialization)
- **Correct output** (mathematically identical to standard attention)

But the *constant-factor overhead* of Python-level loops and unfused operations overwhelms
the memory-access savings.

**Practical takeaway:** Use `torch.nn.functional.scaled_dot_product_attention`, which
dispatches to FlashAttention-2 (or memory-efficient attention) as a fused kernel under the hood.

## 6. PyTorch's Built-in Flash Attention

Since PyTorch 2.0, `torch.nn.functional.scaled_dot_product_attention` automatically
selects the best backend:

| Backend | Requirements | Notes |
|---------|-------------|-------|
| FlashAttention-2 | CUDA, bf16/fp16 | Fastest, IO-aware |
| Memory-efficient (xFormers) | CUDA | Good fallback |
| Math | Any device | Standard attention, always available |

On CUDA with float16/bfloat16, it will use the FlashAttention kernel. On CPU or MPS,
it falls back to the math (standard) implementation but is still optimized.

You can check which backend is used with `torch.backends.cuda.sdp_kernel()` (CUDA only).

In [ ]:
def pytorch_sdpa(Q, K, V):
    """PyTorch 2.0+ fused scaled dot-product attention."""
    return F.scaled_dot_product_attention(Q, K, V)


# Verify it matches our implementations
out_sdpa = pytorch_sdpa(Q, K, V)
max_diff_sdpa = (out_naive - out_sdpa).abs().max().item()
matches_sdpa = torch.allclose(out_naive, out_sdpa, atol=1e-5)

print(f"Max absolute difference (naive vs SDPA): {max_diff_sdpa:.2e}")
print(f"Outputs match (atol=1e-5): {matches_sdpa}")
print(f"\nDevice: {device}")
if device == 'cuda':
    print("SDPA will use FlashAttention-2 or memory-efficient backend")
else:
    print("SDPA will use math (standard) backend on this device")

## 7. Benchmarks

We benchmark all three implementations across sequence lengths to measure:
- **Runtime** (wall-clock time in milliseconds)
- **Memory** (theoretical attention matrix size)

**Parameters:** $d = 64$, batch size $= 4$, sequence lengths $= [128, 256, 512, 1024, 2048, 4096]$

In [ ]:
def benchmark_fn(fn, Q, K, V, num_warmup=3, num_trials=10):
    """Benchmark a function, return median time in milliseconds."""
    # Warmup
    for _ in range(num_warmup):
        _ = fn(Q, K, V)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    for _ in range(num_trials):
        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = fn(Q, K, V)
        if device == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append((end - start) * 1000)  # ms
    
    return np.median(times)


print("Benchmark utilities ready.")

In [ ]:
seq_lengths = [128, 256, 512, 1024, 2048]
d = 64
batch_size = 4

# Skip flash_attention (pure Python) for very long sequences — it's too slow
flash_max_N = 1024

results = {'seq_len': [], 'naive_ms': [], 'flash_ms': [], 'sdpa_ms': []}

for N in seq_lengths:
    Q = torch.randn(batch_size, N, d, device=device)
    K = torch.randn(batch_size, N, d, device=device)
    V = torch.randn(batch_size, N, d, device=device)
    
    # Naive attention
    t_naive = benchmark_fn(naive_attention, Q, K, V)
    
    # Flash attention (pure Python) — skip if N too large
    if N <= flash_max_N:
        t_flash = benchmark_fn(flash_attention, Q, K, V, num_warmup=1, num_trials=3)
    else:
        t_flash = float('nan')
    
    # PyTorch SDPA
    t_sdpa = benchmark_fn(pytorch_sdpa, Q, K, V)
    
    results['seq_len'].append(N)
    results['naive_ms'].append(t_naive)
    results['flash_ms'].append(t_flash)
    results['sdpa_ms'].append(t_sdpa)
    
    flash_str = f"{t_flash:.2f} ms" if not math.isnan(t_flash) else "skipped (too slow)"
    print(f"N={N:5d} | Naive: {t_naive:8.2f} ms | Flash (Python): {flash_str:>20s} | SDPA: {t_sdpa:8.2f} ms")

df = pd.DataFrame(results)
print("\n")
print(df.to_string(index=False))

### Theoretical Memory Analysis

| Sequence Length | Standard Attention ($2N^2 \times 4$ bytes) | Flash Attention ($O(N \times d)$ bytes) |
|:-:|:-:|:-:|
| 128 | 128 KB | 32 KB |
| 256 | 512 KB | 64 KB |
| 512 | 2 MB | 128 KB |
| 1024 | 8 MB | 256 KB |
| 2048 | 32 MB | 512 KB |
| 4096 | 128 MB | 1 MB |

Standard attention memory grows **quadratically** ($N^2$). Flash Attention grows **linearly** ($N$).
At $N = 4096$, standard attention uses **128x** more memory for the attention matrix alone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Runtime vs Sequence Length ---
ax = axes[0]
ax.plot(results['seq_len'], results['naive_ms'], 'o-', label='Naive Attention', linewidth=2)
flash_valid = [(n, t) for n, t in zip(results['seq_len'], results['flash_ms']) if not math.isnan(t)]
if flash_valid:
    ax.plot([x[0] for x in flash_valid], [x[1] for x in flash_valid], 's-', label='Flash (Python)', linewidth=2)
ax.plot(results['seq_len'], results['sdpa_ms'], '^-', label='PyTorch SDPA', linewidth=2)
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Runtime vs Sequence Length', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 2: Memory Usage (Theoretical) ---
ax = axes[1]
N_range = np.array([128, 256, 512, 1024, 2048, 4096])
standard_mem = 2 * N_range**2 * 4 / 1024**2   # MB
flash_mem = N_range * d * 4 / 1024**2            # MB
ax.plot(N_range, standard_mem, 'o-', label='Standard ($O(N^2)$)', linewidth=2, color='tab:red')
ax.plot(N_range, flash_mem, 's-', label='Flash ($O(N)$)', linewidth=2, color='tab:green')
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('Attention Matrix Memory', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 3: Speedup of SDPA over Naive ---
ax = axes[2]
speedup = [n / s for n, s in zip(results['naive_ms'], results['sdpa_ms'])]
ax.bar(range(len(results['seq_len'])), speedup, color='tab:blue', alpha=0.7)
ax.set_xticks(range(len(results['seq_len'])))
ax.set_xticklabels(results['seq_len'])
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Speedup (x)', fontsize=12)
ax.set_title('SDPA Speedup over Naive', fontsize=14)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Reusable Implementation

All three attention functions are also available in `src/models/attention.py` for reuse in other notebooks:

```python
from src.models.attention import naive_attention, flash_attention, pytorch_sdpa
```

In [ ]:
# Verify the src/ imports work
from src.models.attention import naive_attention as na, flash_attention as fa, pytorch_sdpa as ps

Q_test = torch.randn(1, 16, 8, device=device)
K_test = torch.randn(1, 16, 8, device=device)
V_test = torch.randn(1, 16, 8, device=device)

assert torch.allclose(na(Q_test, K_test, V_test), fa(Q_test, K_test, V_test, block_size=4), atol=1e-5)
assert torch.allclose(na(Q_test, K_test, V_test), ps(Q_test, K_test, V_test), atol=1e-5)
print("All src/ imports verified.")

## 9. Key Takeaways

1. **Standard attention is memory-bound.** The $N \times N$ attention matrix requires $O(N^2)$ memory and multiple HBM round trips.

2. **Flash Attention tiles the computation.** By processing blocks of $Q$, $K$, $V$ that fit in SRAM, it never materializes the full $N \times N$ matrix.

3. **Online softmax enables incremental computation.** Running max and running sum statistics allow correct softmax across blocks without seeing all scores at once.

4. **The speedup comes from the memory hierarchy.** SRAM is ~10x faster than HBM. A fused CUDA kernel keeps everything in SRAM; our Python version can't do that.

5. **In practice: use `F.scaled_dot_product_attention`.** It dispatches to FlashAttention-2 automatically on CUDA with float16/bfloat16.

6. **Complexity comparison:**

| | Standard | Flash Attention |
|---|---|---|
| HBM accesses | $O(N^2 d)$ | $O(N^2 d^2 / M)$ |
| Memory | $O(N^2)$ | $O(N)$ |
| Output | Exact | Exact (same result) |

### Further Reading

- **FlashAttention-2** (Dao, 2023): Better parallelism and work partitioning, 2x speedup over FlashAttention
- **FlashAttention-3** (Shah et al., 2024): H100-specific optimizations using WGMMA and TMA
- PyTorch docs: [`torch.nn.functional.scaled_dot_product_attention`](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

## 10. Run on GPU (Modal)

Run the attention benchmarks on a remote NVIDIA GPU via [Modal](https://modal.com).
On CUDA, `F.scaled_dot_product_attention` dispatches to the real FlashAttention-2 kernel,
showing dramatic speedups over naive attention.

```bash
# One-time setup:
pip install modal
modal token set
```

In [ ]:
from src.infra.modal_runner import run_attention_benchmark

gpu_results = run_attention_benchmark.remote(
    seq_lengths=[128, 256, 512, 1024, 2048, 4096],
    d=64,
    batch_size=4,
)

print(f"Benchmarked on: {gpu_results['gpu_name']}")
for i, N in enumerate(gpu_results["seq_lengths"]):
    naive = gpu_results["naive_ms"][i]
    sdpa = gpu_results["sdpa_ms"][i]
    speedup = naive / sdpa if sdpa > 0 else float("inf")
    print(f"N={N:5d} | Naive: {naive:8.2f} ms | SDPA: {sdpa:8.2f} ms | Speedup: {speedup:.1f}x")